# Lesson 02 - Exploring Microsoft Agent Framework

The **Microsoft Agent Framework (MAF)** is a unified framework for building AI agents. It provides a clean, composable architecture with four core building blocks:

- **Client** – connects to an AI model endpoint and handles communication
- **Agent** – wraps a client with instructions and tool definitions
- **Tools** – extend agent capabilities with custom functions the model can call
- **Session** – maintains conversation history for multi-turn interactions

In this lesson, we'll build a **travel booking agent** that checks destination availability using these concepts.


## Asennus


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## Agenttikehyksen arkkitehtuurin ymmärtäminen

Microsoft Agent Framework noudattaa kerroksellista arkkitehtuuria:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Asiakas** – `FoundryChatClient` yhdistää Azure OpenAI -ympäristöön. Se hoitaa todentamisen, pyynnön muotoilun ja vastauksen jäsentämisen.
2. **Agentti** – Luodaan asiakkaasta `provider.create_agent()`-metodilla. Agentti yhdistää mallin käyttöoikeuden järjestelmäkehotteeseen ja työkaluihin.
3. **Työkalut** – Python-funktioita, jotka on koristeltu `@tool`:lla ja joita agentti voi kutsua suorittaakseen toimintoja tai hakeakseen tietoa.
4. **Istunto** – `AgentSession`-objekti (luodaan `agent.create_session()`-metodilla), joka tallentaa keskusteluhistorian mahdollistaen monivuorovaikutteisen keskustelun, jossa agentti muistaa aiemman kontekstin.

Rakennetaan jokainen kerros vaihe vaiheelta.


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Työkalujen lisääminen @tool-koristetta käyttäen

Työkalut antavat agenteille mahdollisuuden suorittaa toimintoja tekstin generoinnin lisäksi. `@tool`-koriste muuntaa tavallisen Python-funktion agentin kutsuttavaksi.

Tärkeimmät kohdat:
- Käytä `Annotated[type, "description"]` jotta malli ymmärtää kunkin parametrin.
- Docstring toimii työkalun kuvauksena, jonka malli näkee.
- `approval_mode="never_require"` tarkoittaa, että työkalu suoritetaan automaattisesti ilman käyttäjän vahvistusta.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Agentin luominen työkaluilla

Nyt yhdistämme asiakkaan, ohjeet ja työkalut agentiksi. `instructions` toimivat järjestelmäkehotteena — ne määrittävät agentin persoonan ja käyttäytymisen.


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Monipuoliset keskustelut istuntojen avulla

`AgentSession` (luotu `agent.create_session()`-metodilla) pitää kirjaa kaikista keskustelun viesteistä. Antamalla saman istunnon jokaiselle `agent.run()` -kutsulle, agentilla on pääsy koko keskusteluhistoriaan ja se voi viitata aiempiin viesteihin.

Annamme `tools=[check_destination_availability]`, jotta agentti voi kutsua saatavuustarkistustamme jokaisella kierroksella.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Yhteenveto

Tässä oppitunnissa tutustuit Microsoft Agent Frameworkin neljään pilariin:

| Käsite | Mitä opit |
|---------|------------------|
| **Asiakas** | `FoundryChatClient` yhdistää Azure OpenAI:hin tunnistetietopohjaisella todennuksella |
| **Agentti** | `provider.create_agent()` yhdistää malliyhteyden ohjeisiin ja nimeen |
| **Työkalut** | `@tool`-koristetta käytetään Python-funktioiden esilletuomiseen agentin kutsuttavaksi |
| **Istunto** | `agent.create_session()` ylläpitää keskusteluhistoriaa useiden vuorojen ajan |

Nämä rakennuspalikat muodostavat yhdessä agenteja, jotka voivat käydä luonnollisia keskusteluja, kutsua ulkoisia toimintoja ja ylläpitää kontekstia — perusta kehittyneemmille agenttimalleille myöhemmissä oppitunneissa.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, otathan huomioon, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäiskielellä on virallinen lähde. Tärkeissä asioissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
